# NatyaVeda — 03 Mudra Analysis
Inspect hand/finger keypoints and mudra detection across dance forms.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.feature_extraction.hand_extractor import HandFrame, MudraRecognizer, HAND_LANDMARK_NAMES

recognizer = MudraRecognizer()

# Load hand features
d = np.load('../data/processed/bharatanatyam/sample_seg000.npz')
kpts = d['keypoints']   # [T, 133, 3]

# Build HandFrames from RTMW wholebody kpts
mudra_labels = []
for t in range(min(100, len(kpts))):
    hf = HandFrame(
        left_hand=kpts[t, 91:112],
        right_hand=kpts[t, 112:133],
        left_confidence=kpts[t, 91:112, 2].mean(),
        right_confidence=kpts[t, 112:133, 2].mean(),
    )
    mudra, conf = recognizer.predict(hf, hand='right')
    mudra_labels.append((mudra, conf))

# Count most frequent mudras
from collections import Counter
counts = Counter([m for m, c in mudra_labels if c > 0.4])
print('Top mudras detected:')
for mudra, count in counts.most_common(10):
    print(f'  {mudra:20s}: {count}')

In [ ]:
# Plot finger angles over time for a single clip
angles_over_time = []
for t in range(min(100, len(kpts))):
    hf = HandFrame(
        left_hand=kpts[t, 91:112],
        right_hand=kpts[t, 112:133],
        left_confidence=kpts[t, 91:112, 2].mean(),
        right_confidence=kpts[t, 112:133, 2].mean(),
    )
    angles_over_time.append(hf.finger_angles('right'))

angles = np.array(angles_over_time)  # [T, 5]
fingers = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']

plt.figure(figsize=(12, 4))
for i, name in enumerate(fingers):
    plt.plot(np.degrees(angles[:, i]), label=name)
plt.xlabel('Frame'); plt.ylabel('Angle (degrees)')
plt.title('Right Hand Finger Flexion Angles')
plt.legend(); plt.tight_layout(); plt.show()